# Ridge pool (return lags only)
Ridge model with **only log-return lags** (ret_lag_1..5). No volume, RSI, MACD, VIX, month, or Fear & Greed. Target = 7-step log returns; convert to price via `p0 * exp(cumsum(pred_log_returns))`. Inputs scaled with StandardScaler.

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.multioutput import MultiOutputRegressor

REPO_ROOT = Path.cwd().parent.parent
BACKEND_DIR = REPO_ROOT / "backend"
sys.path.insert(0, str(BACKEND_DIR))
sys.path.insert(0, str(Path.cwd()))

from _pool_common import (
    load_pool_data,
    build_pooled_train_stack,
    compute_metrics_averaged_over_windows,
    metrics_to_parquet,
    TEST_SIZE,
    FORECAST_HORIZON,
    ROLLING_STEP,
    MIN_TRAIN_STACK,
    ARTIFACTS_DIR,
    TICKERS,
)

LAG_RETURNS = 5
import json
_ridge_core_path = ARTIFACTS_DIR / "best_ridge_core_params.json"
if _ridge_core_path.exists():
    with open(_ridge_core_path) as f:
        _r = json.load(f)
    RIDGE_ALPHA = _r.get("alpha", 1.0)
else:
    RIDGE_ALPHA = 1.0

In [2]:
def build_feature_df(grp: pd.DataFrame):
    """Features: log-return lags only (ret_lag_1..5). Target = next 7 log returns."""
    df = grp.sort_values("timestamp").copy()
    df["close"] = df["close"].astype(float)
    df["log_return"] = np.log(df["close"] / df["close"].shift(1))
    for i in range(1, LAG_RETURNS + 1):
        df[f"ret_lag_{i}"] = df["log_return"].shift(i)
    for h in range(1, FORECAST_HORIZON + 1):
        df[f"target_{h}"] = df["log_return"].shift(-h)
    feature_cols = [f"ret_lag_{i}" for i in range(1, LAG_RETURNS + 1)]
    target_cols = [f"target_{h}" for h in range(1, FORECAST_HORIZON + 1)]
    base_cols = ["timestamp", "close", "log_return"] + feature_cols + target_cols
    out = df[[c for c in base_cols if c in df.columns]].copy()
    return out.dropna(), feature_cols, target_cols


def train_global_ridge(stacked: pd.DataFrame, horizon: int):
    """Train one Ridge (MultiOutputRegressor) on pooled data. Returns dict for predict_ridge_global."""
    pooled = build_pooled_train_stack(stacked, TEST_SIZE, MIN_TRAIN_STACK)
    if pooled.empty:
        return None
    feat_dfs = []
    for sym in pooled["symbol"].unique():
        grp = pooled[pooled["symbol"] == sym].copy()
        try:
            feat_df, feature_cols, target_cols = build_feature_df(grp)
        except Exception:
            continue
        if len(feat_df) < MIN_TRAIN_STACK + horizon:
            continue
        feat_dfs.append(feat_df)
    if not feat_dfs:
        return None
    pooled_feat = pd.concat(feat_dfs, ignore_index=True)
    X = pooled_feat[feature_cols].values.astype(np.float64)
    y = pooled_feat[target_cols].values.astype(np.float64)
    scaler = StandardScaler()
    X_s = scaler.fit_transform(X)
    ridge = MultiOutputRegressor(Ridge(alpha=RIDGE_ALPHA, random_state=42))
    ridge.fit(X_s, y)
    return {"model": ridge, "scaler": scaler, "feature_cols": feature_cols}


def predict_ridge_global(context_df: pd.DataFrame, horizon: int, global_ridge: dict) -> list:
    """Predict 7 price steps using pre-trained global Ridge (model outputs log returns; compound to price)."""
    if global_ridge is None:
        return []
    try:
        feat_df, feature_cols, _ = build_feature_df(context_df)
    except Exception:
        return []
    if len(feat_df) < 1:
        return []
    X = feat_df[feature_cols].values.astype(np.float64)
    X_s = global_ridge["scaler"].transform(X)
    last_row = X_s[-1:]
    pred_log_returns = global_ridge["model"].predict(last_row).ravel()
    p0 = float(context_df["close"].iloc[-1])
    prices = p0 * np.exp(np.cumsum(pred_log_returns))
    return [float(p) for p in prices[:horizon]]

In [3]:
stacked = load_pool_data(with_vix=False, with_volume=False, use_fixed=True)
print(stacked.groupby("symbol").size())
stacked.head(10)

symbol
AAPL     1256
AMZN     1256
GOOGL    1256
JNJ      1256
JPM      1256
MSFT     1256
NVDA     1256
SPY      1256
WMT      1256
XOM      1256
dtype: int64


,timestamp,symbol,close
0,2021-03-01,AAPL,127.790001
1,2021-03-02,AAPL,125.120003
2,2021-03-03,AAPL,122.059998
3,2021-03-04,AAPL,120.129997
4,2021-03-05,AAPL,121.419998
5,2021-03-08,AAPL,116.360001
6,2021-03-09,AAPL,121.089996
7,2021-03-10,AAPL,119.980003
8,2021-03-11,AAPL,121.959999
9,2021-03-12,AAPL,121.029999


In [4]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.metrics import make_scorer

def mean_mae_multioutput(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))

neg_mean_mae_scorer = make_scorer(mean_mae_multioutput, greater_is_better=False)

pooled = build_pooled_train_stack(stacked, TEST_SIZE, MIN_TRAIN_STACK)
feat_dfs = []
for sym in pooled["symbol"].unique():
    grp = pooled[pooled["symbol"] == sym].copy()
    try:
        feat_df, feature_cols, target_cols = build_feature_df(grp)
    except Exception:
        continue
    if len(feat_df) < MIN_TRAIN_STACK + FORECAST_HORIZON:
        continue
    feat_dfs.append(feat_df)
pooled_feat = pd.concat(feat_dfs, ignore_index=True)
pooled_feat = pooled_feat.sort_values("timestamp").reset_index(drop=True)
X_rs = pooled_feat[feature_cols].values.astype(np.float64)
y_rs = pooled_feat[target_cols].values.astype(np.float64)

pipe_ridge = Pipeline([
    ("scaler", StandardScaler()),
    ("multi", MultiOutputRegressor(Ridge(random_state=42)))
])
param_dist = {"multi__estimator__alpha": [1e-4, 0.001, 0.01, 0.1, 1.0, 10.0]}
tscv = TimeSeriesSplit(n_splits=5)
search = RandomizedSearchCV(pipe_ridge, param_dist, n_iter=6, cv=tscv, scoring=neg_mean_mae_scorer, random_state=42, n_jobs=-1, verbose=1)
search.fit(X_rs, y_rs)

best = {k.replace("multi__estimator__", ""): float(v) for k, v in search.best_params_.items()}
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
with open(ARTIFACTS_DIR / "best_ridge_core_params.json", "w") as f:
    json.dump(best, f, indent=2)
RIDGE_ALPHA = best["alpha"]
print("Best Mean MAE (CV):", -search.best_score_)
print("Best RIDGE_ALPHA:", RIDGE_ALPHA)
print("Saved:", ARTIFACTS_DIR / "best_ridge_core_params.json")

Fitting 5 folds for each of 6 candidates, totalling 30 fits
Best Mean MAE (CV): 0.013162453681329811
Best RIDGE_ALPHA: 10.0
Saved: C:\capstone_project_unfc\model\experiments-pool\artifacts\best_ridge_core_params.json


In [5]:
global_ridge = train_global_ridge(stacked, FORECAST_HORIZON)
print("Global Ridge (core) trained on", len(build_pooled_train_stack(stacked, TEST_SIZE, MIN_TRAIN_STACK)), "pooled train rows.")

Global Ridge (core) trained on 11960 pooled train rows.


In [6]:
if global_ridge is not None:
    multi = global_ridge["model"]
    cols = global_ridge["feature_cols"]
    coefs = np.array([multi.estimators_[h].coef_ for h in range(len(multi.estimators_))])
    df_coef = pd.DataFrame(coefs.T, index=cols, columns=[f"h{h+1}" for h in range(coefs.shape[0])])
    print("Ridge coefficients (return lags only):")
    display(df_coef)
else:
    print("No trained model. Run the training cell first.")

Ridge coefficients (return lags only):


,h1,h2,h3,h4,h5,h6,h7
ret_lag_1,-0.000015,-0.000815,-0.000127,0.000064,0.000213,-0.000222,-0.000145
ret_lag_2,-0.000808,-0.000119,0.000061,0.000219,-0.000218,-0.000146,0.000370
ret_lag_3,-0.000109,0.000054,0.000201,-0.000208,-0.000161,0.000379,-0.000348
ret_lag_4,0.000060,0.000148,-0.000201,-0.000167,0.000383,-0.000359,0.000246
ret_lag_5,0.000152,-0.000199,-0.000165,0.000388,-0.000360,0.000243,-0.000107


In [7]:
model_name = "ridge_core"
all_preds = []
for sym in TICKERS:
    grp = stacked[stacked["symbol"] == sym].copy()
    if grp.empty:
        continue
    grp = grp.sort_values("timestamp").reset_index(drop=True)
    prices = grp.set_index("timestamp")["close"].astype(float).dropna()
    n = len(prices)
    if n < TEST_SIZE + MIN_TRAIN_STACK:
        continue
    split_idx = n - TEST_SIZE
    test_index = prices.index[split_idx:]
    test_values = prices.values[split_idx:]
    preds = []
    window_ix = 0
    start = 0
    while start + FORECAST_HORIZON <= TEST_SIZE:
        context_df = grp.iloc[: split_idx + start][["timestamp", "close"] + [c for c in ["volume"] if c in grp.columns]].copy()
        if len(context_df) < MIN_TRAIN_STACK:
            start += ROLLING_STEP
            continue
        price_list = predict_ridge_global(context_df, FORECAST_HORIZON, global_ridge)
        if not price_list or len(price_list) < FORECAST_HORIZON:
            start += ROLLING_STEP
            window_ix += 1
            continue
        for h in range(FORECAST_HORIZON):
            idx = start + h
            ts = test_index[idx]
            y_true = float(test_values[idx])
            y_pred = float(price_list[h])
            preds.append({"timestamp": ts, "y_true": y_true, "y_pred": y_pred, "window_ix": window_ix})
        window_ix += 1
        start += ROLLING_STEP
    if preds:
        pred_df = pd.DataFrame(preds)
        pred_df["symbol"] = sym
        all_preds.append(pred_df)

pred_stack = pd.concat(all_preds, ignore_index=True) if all_preds else pd.DataFrame(columns=["timestamp", "y_true", "y_pred", "window_ix", "symbol"])
print(pred_stack.groupby("symbol").size() if not pred_stack.empty else "No predictions.")
pred_stack.head()

symbol
AAPL     56
AMZN     56
GOOGL    56
JNJ      56
JPM      56
MSFT     56
NVDA     56
SPY      56
WMT      56
XOM      56
dtype: int64


,timestamp,y_true,y_pred,window_ix,symbol
0,2025-12-02,286.190002,283.526253,0,AAPL
1,2025-12-03,284.149994,283.790073,0,AAPL
2,2025-12-04,280.700012,283.997651,0,AAPL
3,2025-12-05,278.779999,284.113331,0,AAPL
4,2025-12-08,277.890015,284.414193,0,AAPL


In [8]:
metrics_rows = []
for sym in pred_stack["symbol"].unique():
    sub = pred_stack[pred_stack["symbol"] == sym]
    m = compute_metrics_averaged_over_windows(sub)
    metrics_rows.append({"model": model_name, "symbol": sym, **m})
m_overall = compute_metrics_averaged_over_windows(pred_stack)
metrics_rows.append({"model": model_name, "symbol": "overall", **m_overall})

metrics_df = pd.DataFrame(metrics_rows)
print(metrics_df.to_string())
metrics_to_parquet(metrics_rows, ARTIFACTS_DIR / "metrics_ridge_core_pool.parquet")
print("Saved:", ARTIFACTS_DIR / "metrics_ridge_core_pool.parquet")

         model   symbol        MAE       RMSE    MAPE_%
0   ridge_core     AAPL   6.894274   7.753649  2.617318
1   ridge_core     MSFT  11.475957  12.452072  2.607055
2   ridge_core    GOOGL   8.152136   9.196063  2.568207
3   ridge_core     AMZN   9.032965   9.998113  4.025503
4   ridge_core      JPM   7.221098   8.043510  2.306864
5   ridge_core      JNJ   3.892726   4.326573  1.748044
6   ridge_core      WMT   2.492071   2.822500  2.075097
7   ridge_core      SPY   6.653369   7.435529  0.971168
8   ridge_core      XOM   4.563919   4.907466  3.335518
9   ridge_core     NVDA   4.207147   4.898985  2.308696
10  ridge_core  overall   6.458566   8.451896  2.456347
Saved: C:\capstone_project_unfc\model\experiments-pool\artifacts\metrics_ridge_core_pool.parquet
